In [0]:
%pip install -qq --upgrade "mlflow[databricks]>=3.1.0"
dbutils.library.restartPython()

In [0]:
try:  experiment_name_value = dbutils.widgets.get("mlflow_experiment_name")
except:
    experiment_name_value = None
if experiment_name_value is None:
    dbutils.widgets.text("experiment_name", "")

    dbutils.widgets.text("experiment_name", "")
mlflow_experiment_name = dbutils.widgets.get("mlflow_experiment_name")
print(f'mlflow experiment : {mlflow_experiment_name}')


## Agent evaluation via existing trace

In [0]:
import asyncio
import logging
import mlflow 

from dotenv import load_dotenv
from mlflow.genai.agent_server import get_invoke_function
from mlflow.types.responses import ResponsesAgentRequest
from mlflow.genai.scorers import (
    Completeness,
    ConversationCompleteness,
    ConversationalSafety,
    ToolCallEfficiency,
    RelevanceToQuery,
    Safety,
    UserFrustration,
    Correctness
) 

# Load environment variables from .env if it exists
load_dotenv(dotenv_path=".env", override=True)
logging.getLogger("mlflow.utils.autologging_utils").setLevel(logging.ERROR)

mlflow.set_experiment(experiment_name=mlflow_experiment_name)

eval_traces = mlflow.search_traces(experiment_ids=[mlflow_experiment_id], max_results=10000, filter_string="status='OK'")
print(f'total number of traces : {len(eval_traces)}')
display(eval_traces)

In [0]:
def evaluate():
    mlflow.genai.evaluate(
        data=eval_traces,
        scorers=[
            Completeness(),
            # ConversationCompleteness(), # only relevant for multi-turn conversation 
            # ConversationalSafety(), # only relevant for multi-turn conversation 
            # UserFrustration(), # only relevant for multi-turn conversation 
            RelevanceToQuery(),
            Safety(),
            ToolCallEfficiency(),
            Correctness()
        ],
    )
evaluate()


## Agent Evaluation via Calling App API

In [0]:

@mlflow.trace
def predict_fn(input: list[dict], **kwargs) -> dict:
    """Call the deployed Databricks App agent via HTTP."""
    # Step 1: Get your app's OAuth client ID
    w = WorkspaceClient()
    app_name = "agent-test-app"   
    app_client_id = w.apps.get(app_name).oauth2_app_client_id

    # Step 2: Get the notebook's internal token
    notebook_token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    # Step 3: Exchange it for an audience-scoped OAuth token
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    token_response = requests.post(
        url=f"https://{workspace_url}/oidc/v1/token",
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
            "subject_token": notebook_token,
            "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
            "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
            "scope": "all-apis",
            "audience": app_client_id,
        },
    )
    audience_token = token_response.json()["access_token"]

    response = requests.post(
        f"{APP_URL}/responses",  # Adjust path based on your agent's API
        headers={
            "Authorization": f"Bearer {audience_token}",
            "Content-Type": "application/json",
        },
        json={"input": input},
    )
    response.raise_for_status()
    return response.json()

eval_data = [
    {"inputs": {"input": [{"role": "user", "content": "How much revenue loss is tied with Return or refund and what does our refund policy says about it ? "}]}},
]

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[RelevanceToQuery(), 
             Safety(), 
             ToolCallEfficiency()
             ],
)
results

In [0]:
from mlflow.genai import datasets, evaluate
from mlflow.genai.scorers import (
    Safety,
    RelevanceToQuery,
    Guidelines,
    Correctness
)

EVAL_DATASET_NAME='ai_workshop_series_catalog.education.education'

# Get the dataset.
dataset = datasets.get_dataset(EVAL_DATASET_NAME)
display(dataset.to_df())

# Run the evaluation.
# Select scorers relevant to your use case. See all available
# scorers: https://docs.databricks.com/mlflow3/genai/eval-monitor/predefined-judge-scorers
evaluate(
  data=dataset,
  scorers=[
    Safety(),
    RelevanceToQuery(),
    Correctness(),
    Guidelines(name="conciseness", guidelines="Responses must be concise."),
  ],
)